# Parametric Statistical Tests
In both industry and academia, data-driven decision-making is inextricably linked to the application of statistical tests. Parametric tests, which are based on certain distributional assumptions (such as normality and homogeneity of variance), are one of the primary methods for analyzing quantitative data.

This portfolio was designed to demonstrate the application of various parametric tests using Python and public datasets from the Seaborn library. The datasets used include:
1. Tips Dataset → for analyzing restaurant customer tips.
2. Iris Dataset → for testing average differences between flower species.
3. Penguins Dataset → for measuring relationships and morphological differences between penguin species.

The statistical tests presented include:
1. Assumption Tests: Normality (Shapiro–Wilk) and Homogeneity of Variance (Levene's Test).
2. t-Tests: One-Sample, Independent Samples, and Paired Samples.
3. ANOVA (One-Way): for mean differences between more than two groups.
4. Pearson Correlation: to test linear relationships between quantitative variables.
5. MANOVA (Multivariate ANOVA): to test multivariate mean differences between groups.
6. Chi-Square Test: to test the distribution of categorical data or the conformity of observation frequencies to the expected distribution.

With structured explanations and in-depth interpretation of results, this portfolio showcases not only code and output but also analytical narratives that can support understanding in the context of data science, market research, and industry.

# Import Library & Dataset

In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA

# Load datasets
tips = sns.load_dataset("tips")
iris = sns.load_dataset("iris")
penguins = sns.load_dataset("penguins")

In [ ]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [ ]:
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [ ]:
penguins.isnull().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [ ]:
penguins.dropna(inplace=True)

In [ ]:
penguins.isnull().sum()

,0
species,0
island,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,0


# Test for Categorical Relationship (Chi-Square Test)
* Research Question\
Is there a significant relationship between sepal length (short vs. long) and iris species?
* To answer this question, we used the Chi-Square test of independence, which is suitable for categorical data.
* Hypothesis:

  H₀: The distribution of sepal length is independent of species (no relationship).

  H₁: The distribution of sepal length is dependent on species (there is a relationship).

Binning sepal_length into categories

In [2]:
iris["sepal_cat"] = pd.cut(iris["sepal_length"],
                           bins=[0, 5.5, 8],
                           labels=["Short", "Long"])

Create a contingency table

In [3]:
contingency = pd.crosstab(iris["sepal_cat"], iris["species"])
print(contingency)

species    setosa  versicolor  virginica
sepal_cat                               
Short          47          11          1
Long            3          39         49


Chi-Square Test

In [4]:
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print("Expected:\n", np.round(expected,2))
print("Chi-square:", np.round(chi2,2))
print("p-value:", p)
print("Degrees of freedom:", dof)

Expected:
 [[19.67 19.67 19.67]
 [30.33 30.33 30.33]]
Chi-square: 98.12
p-value: 4.940452295286229e-22
Degrees of freedom: 2


In [5]:
# χ² critical
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, dof)
print("Chi-square critical value:", round(chi2_critical,2))

Chi-square critical value: 5.99


* Chi-Square Test Results\
  Chi-Square statistic (χ²) = 98.12\
  Degrees of Freedom (df) = 2\
  p-value = 4.94e-22\
  Critical χ² (α = 0.05, df = 2) ≈ 5.99
* Interpretation
  * Because the calculated χ² = 98.12 is greater than the critical χ² = 5.99, H₀ is rejected.
  * With p < 0.05, this result also supports the rejection of H₀.
  * This means there is a significant relationship between sepal length and iris species. In other words, the distribution of sepal lengths differs significantly between species.
* Conclusion\
The Chi-Square test shows that iris species significantly influences sepal length categories. This finding makes sense because each iris species has distinct morphological characteristics.

# Assumption Testing (Normality & Homogeneity)
* Research Question
Is there a difference in the average tips given by male and female customers?
* Before proceeding to the t-test or ANOVA, two key assumptions must be met:
  1. Normality → the data in each group must be normally distributed.
  2. Homogeneity of variance → the variances between the groups being compared must be equal.
* In this example, the distribution of tips based on customer gender (Male and Female) from the tips dataset is tested.

In [ ]:
male_tips = tips[tips["sex"] == "Male"]["tip"]
female_tips = tips[tips["sex"] == "Female"]["tip"]

## Shapiro-Wilk normality test

In [ ]:
print("Shapiro-Wilk Male:", stats.shapiro(male_tips))
print("Shapiro-Wilk Female:", stats.shapiro(female_tips))

Shapiro-Wilk Male: ShapiroResult(statistic=np.float64(0.8758690617789388), pvalue=np.float64(3.7084828294513925e-10))
Shapiro-Wilk Female: ShapiroResult(statistic=np.float64(0.9567775372726819), pvalue=np.float64(0.005448280473692281))


* Normality Test Results (Shapiro–Wilk Test)\
Male: statistic = 0.876, p-value = 3.71e-10\
Female: statistic = 0.957, p-value = 0.0054
* Interpretation
  - H₀: The data comes from a normal distribution.
  - Since p < 0.05 for both groups, H₀ is rejected.
  - This means that neither the male nor the female sex ratios are normally distributed.

## Test of homogeneity of variance (Levene's test)

In [ ]:
print("Levene test:", stats.levene(male_tips, female_tips))

Levene test: LeveneResult(statistic=np.float64(1.9909710178779405), pvalue=np.float64(0.1595236359896614))


In [ ]:
# Degrees of freedom
k = 2 # Number of groups
n_total = len(male_tips) + len(female_tips)
df_between = k - 1
df_within = n_total - k

# F-table at alpha = 0.05
f_table = stats.f.ppf(1-0.05, df_between, df_within)

print("F-critical: ",f_table)

F-critical:  3.8801716626689986


* Results of the Homogeneity of Variance Test (Levene's Test)\
F-statistic = 1.991\
F-critical = 3.88\
p-value = 0.160
* Interpretation:
  * H₀: The variances between groups are equal (homogeneous).
  * Because the F-statistic = 1.991 is smaller than the F-critical = 3.88, H₀ is rejected.
  * Because p > 0.05 → H₀ is rejected.
  * This means that the variances between men and women can be considered homogeneous.

* Conclusion
  * The assumption of normality is not met.
  * The assumption of homogeneity of variance is met.
  * Because normality fails, a parametric test (t-test) can still be used if the sample is large due to the Central Limit Theorem.
  * However, for more robust results, a non-parametric test (Mann–Whitney U test) is recommended.

# One-Sample t-test
* Research Question\
Is the average tip given by customers significantly different from $3?

* Parametric Test: One-Sample t-test
* Purpose: To test whether the sample mean is significantly different from a specified value (µ₀).
* Hypothesis:

  H₀: µ = 3 (average tip equals $3)

  H₁: µ ≠ 3 (average tip differs from $3)

In [ ]:
sample = tips["tip"].dropna()
t_stat, p_value = stats.ttest_1samp(sample, 3)
t_stat, p_value

(np.float64(-0.019432641422916876), np.float64(0.9845119176410544))

In [ ]:
print(p_value)

0.9845119176410544


In [ ]:
print(t_stat)

-0.019432641422916876


In [ ]:
# Degrees of freedom
df = len(sample) - 1

# t-table at alpha = 0.05 (two-tailed)
t_table = stats.t.ppf(1-0.025, df)

print("t-critical: ", t_table)

t-critical:  1.9697743954258793


* One-Sample t-test results\
t-statistic = -0.019\
t-critic = 1.967\
p-value = 0.985
* Interpretation:
  * H₀: Average tip = \$3.
  * Because |t-statistic| = 0.19 is smaller than t-critic = -1.967, we fail to reject H₀.
  * With p-value = 0.985 (> 0.05), we fail to reject H₀.
  * This means there is no significant evidence that the average tip differs from \$3.
* Conclusion\
Based on the One-Sample t-test, the average customer tip is not significantly different from \$3. In other words, the data supports that $3 is a reasonable value for the average tip.

# Independent Samples t-test
* Research Question\
Is there a difference in the average tip between male and female customers?
* Parametric Test: Independent Samples t-test
* Purpose: To test whether the averages of two independent groups are significantly different.
* Hypothesis:

  H₀: μₘₐₗₑ = μfₑₘₐₗₑ (men's average tip is the same as women's)

  H₁: μₘₐₗₑ ≠ μfₑₘₐₗₑ (men's average tip is different from women's)

In [ ]:
male = tips[tips["sex"]=="Male"]["tip"].dropna()
female = tips[tips["sex"]=="Female"]["tip"].dropna()

t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)
t_stat, p_value

(np.float64(1.489536377092501), np.float64(0.13780683808650296))

In [ ]:
print(t_stat)

1.489536377092501


In [ ]:
print(p_value)

0.13780683808650296


In [ ]:
# Degrees of freedom (Welch–Satterthwaite)
df = (male.var()/male.count() + female.var()/female.count())**2 / (
((male.var()/male.count())**2)/(male.count()-1) +
((female.var()/female.count())**2)/(female.count()-1)
)

# T-table value at alpha = 0.05 (two-tailed)
t_table = stats.t.ppf(1-0.025, df)

print("t-critical: ",t_table)

t-critical:  1.9710225545069828


* Independent Samples t-test Results\
t-statistic = 1.490\
t-critical = 1.971\
p-value = 0.138
* Interpretation:
  * H₀: There is no difference in average tips between men and women.
  * Because the t-statistic = 1.490 is smaller than the t-critical = 1.971 at α=0.05, H₀ is not rejected.
  * With a p-value = 0.138 (>0.05), H₀ is not rejected.
  * This means there is no significant evidence that average tips differ between men and women.
* Conclusion\
Based on the Independent Samples t-test, there is no significant difference in average tips between men and women. Therefore, customer gender does not statistically affect average tips.

# Paired Samples t-test
* Research Question\
Is there a significant difference between the total bill and tip values ​​(after aligning per transaction)?
* Parametric Test: Paired Samples t-test
* Purpose: To compare the means of two paired measurements (e.g., before–after, or two related variables in the same individual).
* Hypothesis:

  H₀: μᵦᵢₗₗ = μₜᵢₚ (there is no mean difference between total bill and tip)

  H₁: μᵦᵢₗₗ ≠ μₜᵢₚ (there is a mean difference between total bill and tip)

In [ ]:
bill = tips["total_bill"].dropna()
tip = tips["tip"].dropna().iloc[:len(bill)]

t_stat, p_value = stats.ttest_rel(bill, tip)
t_stat, p_value

(np.float64(32.646504518298634), np.float64(8.020018605020848e-91))

In [ ]:
print(t_stat)

32.646504518298634


In [ ]:
print(p_value)

8.020018605020848e-91


In [ ]:
# Degrees of freedom (df = n - 1)
df = len(bill) - 1

# Critical t-value for alpha = 0.05 (two-tailed)
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha/2, df)
print("t-critical (0.05, two-tailed):", t_critical)

t-critical (0.05, two-tailed): 1.9697743954258793


* Paired Samples t-test results\
t-statistic = 32.647\
t-critical = 1.969\
p-value = 8.02e-91
* Interpretation:
  * H₀: The average total bill is equal to the average tip.
  * Because the t-statistic = 32.647 is much greater than the t-critical = 1.969 at α=0.05, reject H₀.
  * With a p-value <0.05, reject H₀.
  * This means there is a very significant difference between the average total bill and tip.
* Conclusion
  The test results show that the average total bill and tip are significantly different. This is reasonable, as the total bill is systematically larger than the tip.

# ANOVA (One-Way)
* Research Question\
Is there a difference in the average sepal length between Iris species (setosa, versicolor, virginica)?
* Parametric Test: One-Way ANOVA
* Purpose: To test whether the means of more than two groups differ significantly.
* Hypothesis:

  H₀: μₛₑₜₒₛₐ = μᵥₑᵣₛᵢcₒₗₒᵣ = μᵥᵢᵣgᵢₙᵢcₐ (there is no difference in the means between species).

  H₁: There is at least one species with a different mean.

In [ ]:
anova = ols("sepal_length ~ species", data=iris).fit()
anova_table = sm.stats.anova_lm(anova, typ=2)
anova_table

,sum_sq,df,F,PR(>F)
species,63.212133,2.0,119.264502,1.669669e-31
Residual,38.956200,147.0,NaN,NaN


In [ ]:
alpha = 0.05
df1 = 2 # degrees of freedom between groups (k - 1)
df2 = 147 # degrees of freedom within groups (N - k)

# Critical F value (F table)
f_crit = stats.f.ppf(1 - alpha, df1, df2)
print("F-critical (alpha=0.05):", f_crit)

F-critical (alpha=0.05): 3.057620651649394


* ANOVA Test Results (One-Way)\
F-statistic = 119.265\
F-critical (α=0.05, df1=2, df2=147) = 3.057\
p-value = 1.67e-31
* Interpretation:
  * H₀: There is no difference in average sepal length between species.
  * Because F-statistic = 119.265 > F-critical = 3.057, reject H₀.
  * With a p-value <0.05, reject H₀.
  * This means there is a significant difference in average sepal length in at least one Iris species.
* Conclusion\
Based on the One-Way ANOVA test, the sepal length of Iris flowers differs significantly between species. To determine which species specifically differ, further testing (post-hoc test, for example, Tukey HSD) is required.

# Pearson Correlation
* Research Question\
Is there a significant linear relationship between bill length and flipper length in penguins?
* Parametric Test: Pearson Correlation
* Purpose: To measure the strength and direction of the linear relationship between two quantitative variables.
* Hypothesis:

  H₀: ρ = 0 (there is no linear correlation between bill length and flipper length).

  H₁: ρ ≠ 0 (there is a significant linear correlation between bill length and flipper length).

In [ ]:
penguins_clean = penguins.dropna(subset=["bill_length_mm","flipper_length_mm"])
corr, p_value = stats.pearsonr(penguins_clean["bill_length_mm"], penguins_clean["flipper_length_mm"])
corr, p_value

(np.float64(0.6530956386670859), np.float64(7.211340708097467e-42))

In [ ]:
print(corr)

0.6530956386670859


In [ ]:
print(p_value)

7.211340708097467e-42


* Pearson Correlation Test Results\
r (correlation) = 0.653\
p-value = 7.21e-42
* Interpretation:
  * The correlation value of r = 0.653 indicates a strong positive relationship between beak length and wing length.
  * With a p-value < 0.05, reject H₀.
  * This means there is a significant linear correlation between the two variables.
* Conclusion\
There is a strong positive relationship between beak length and wing length in penguins. The longer the beak, the longer the wings tend to be.

# MANOVA (Multivariate ANOVA)
* Research Question\
Are there significant differences in bill length and flipper length of penguins based on species (Adelie, Chinstrap, Gentoo)?
* Parametric Test: MANOVA (Multivariate ANOVA)
* Purpose: To test the multivariate mean differences between groups, namely whether the independent variable (species) simultaneously affects several dependent variables (bill length and flipper length).
* Hypothesis:

  H₀: There is no multivariate difference in mean bill length and flipper length between species.

  H₁: There is a multivariate difference in the mean in at least one species.

In [ ]:
penguins_clean = penguins.dropna(subset=["bill_length_mm","flipper_length_mm","species"])
maov = MANOVA.from_formula("bill_length_mm + flipper_length_mm ~ species", data=penguins_clean)
print(maov.mv_test())

                    Multivariate linear model
                                                                  
------------------------------------------------------------------
       Intercept         Value   Num DF  Den DF   F Value   Pr > F
------------------------------------------------------------------
          Wilks' lambda   0.0028 2.0000 329.0000 59140.0033 0.0000
         Pillai's trace   0.9972 2.0000 329.0000 59140.0033 0.0000
 Hotelling-Lawley trace 359.5137 2.0000 329.0000 59140.0033 0.0000
    Roy's greatest root 359.5137 2.0000 329.0000 59140.0033 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
            species         Value  Num DF  Den DF  F Value  Pr > F
------------------------------------------------------------------
              Wilks' lambda 0.0878 4.0000 658.0000 390.5566 0.0000
             Pil

* MANOVA Test Results
  Summary of results for the species factor:
  * Wilks’ Lambda = 0.0878, F(4, 658) = 390.56, p < 0.001
  * Pillai’s Trace = 1.3812, F(4, 660) = 368.29, p < 0.001
  * Hotelling-Lawley Trace = 5.0452, F ≈ 414.55, p < 0.001
  * Roy’s Greatest Root = 3.5342, F ≈ 583.15, p < 0.001
* Interpretation:
  * All multivariate tests (Wilks, Pillai, Hotelling, Roy) yielded p < 0.05.
  * Therefore, reject H₀ → there is a significant multivariate difference in mean bill length and flipper length between species.
* Conclusion\
Penguin species significantly influences body size (beak length and wing length). In other words, penguin species have statistically distinct morphological profiles. Further analysis can be performed using post-hoc tests (e.g., pairwise MANOVA or univariate ANOVA per variable).

# Conclusion
Through a series of parametric statistical tests, this portfolio demonstrates how statistical theory can be applied in practice using Python.

Several key points to conclude:
* Statistical assumptions need to be tested before conducting the main analysis (normality and homogeneity).
* The t-test is useful for simple comparisons of means, whether in one sample, two independent samples, or repeated-measures.
* ANOVA allows for the analysis of mean differences across more than two groups, while MANOVA extends this to multivariate analysis.
* Pearson correlation provides insight into linear relationships between variables.
* The Chi-Square test is an important alternative when dealing with categorical data or when looking to assess the fit of a distribution to theory.
* Analysis results always need to be interpreted in a practical context, not just the p-value.

This portfolio is intended to be a reference that not only showcases technical skills in Python but also emphasizes critical thinking and analytical skills in interpreting statistical test results. Thus, this portfolio can provide a strong foundation for the application of statistics in various fields, from academic research to strategic decision-making in industry.

# Thank You